In [ ]:
from AI4AO import PyramidWFS, PhaseDataset, DeformableMirror
from AI4AO import imshow, imshow_multiple
from mmengine import Config
from kalman.Kalman_tools_numpy import *
import torch
device = "cuda"

import pylab as plt
import numpy as np
import time
from collections import deque

import dao

from AI4AO.paths import DATA_DIR

device = 'cuda'
PATH = DATA_DIR / "Data"

In [ ]:
M2C = np.load(PATH / r"M2C.npy").astype(np.float32)

MODAL_VECTOR_INFERENCE = "/tmp/pyrModesNN.im.shm"
CONTROL_SIGNAL = "/tmp/ctrlModes07.im.shm"
SHM_LOOP_CMD_NN = "/tmp/lpCmdNN.im.shm" # Ajouté
SHM_LOOP_GAIN = "/tmp/lpGain.im.shm"
SHM_LOOP_LEAK = "/tmp/lpLeak.im.shm"

MODAL_VECTOR_INFERENCE_SHM = dao.shm(MODAL_VECTOR_INFERENCE)
CONTROL_SIGNAL_SHM = dao.shm(CONTROL_SIGNAL)
SHM_LOOP_CMD_NN_SHM = dao.shm(SHM_LOOP_CMD_NN)
SHM_LOOP_GAIN_SHM = dao.shm(SHM_LOOP_GAIN)
SHM_LOOP_LEAK_SHM = dao.shm(SHM_LOOP_LEAK)

shm = dao.shm('/tmp/lpCmdNNModalEnable.im.shm')
shm.set_data(np.full_like(np.asarray(shm.get_data()), 0))

nModes = 70

In [ ]:
paramfile = r'/home/rama/rama-dev/foyarzun/CNN/Scripts/Rama_params.py'  # file of experimental parameters


# Config extraction
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']
DMParams = Config.fromfile(paramfile)['DMParams']

In [ ]:
WFSParams["centralObstruction"] = 0.
wfs = PyramidWFS(WFSParams, device)
PATH_WFS = PATH / "RamaWFS.pth"
wfs.LoadCalibration(PATH_WFS)
wfs.eval()


dm = DeformableMirror(WFSParams, DMParams, device)
PATH_DM = PATH / "RamaDM.pth"
dm.LoadCalibration(PATH_DM)

modes = dm(torch.from_numpy(M2C.T).cuda())[:nModes]
wfs.modulation = 0
wfs.BuildReconstructionMatrix(modes)

In [ ]:
# Loop delay: z_k contains the command computed delay frames earlier, u_{k-delay}
delay = 2

z_estimated = MODAL_VECTOR_INFERENCE_SHM.get_data() * 0
Nmodes = MODAL_VECTOR_INFERENCE_SHM.get_data().shape[0]

gain = SHM_LOOP_GAIN_SHM.get_data()[0,0]
leak = SHM_LOOP_LEAK_SHM.get_data()[0,0]

wfs_measurements, dm_commands = [], []

n_steps = 5000
i = 0

while 1:

    if SHM_LOOP_CMD_NN_SHM.get_data()[0,0]:
        z_output = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)

        z_estimated = z_estimated * leak + gain * z_output

        CONTROL_SIGNAL_SHM.set_data(-z_estimated)
        i += 1

    else:
        i = 0
        z_output = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)
        z_estimated *= 0
        CONTROL_SIGNAL_SHM.set_data(z_estimated)


    if i > n_steps * 0.1:
        wfs_measurements.append(np.copy(z_output[:nModes, 0]))
        dm_commands.append(np.copy(z_estimated[:nModes, 0]))

    if i > n_steps: break
    
kalmanData = KalmanData(
    wfs_measurements=np.stack(wfs_measurements).astype(np.float32),
    modal_dm_commands=np.stack(dm_commands).astype(np.float32)
)

In [ ]:
x = estimate_atmospheric_state(kalmanData.wfs_measurements, kalmanData.modal_dm_commands, delay)
A = estimate_A(x)
R = estimate_R(
    wfs.reconstructionMatrix.detach().cpu().numpy(),
    wfs.reference_intensity.detach().cpu().numpy(),
    250*34*34*3.14, 10,
).astype(x.dtype)
Q = estimate_raw_Q(x, A)
# Q = estimate_Q(Q_raw, A, R)
H = np.eye(nModes, dtype=x.dtype)
P0 = A @ np.cov(x.T).astype(x.dtype) @ A.T + Q

In [ ]:
matrices = KalmanMatrices(
    A=A,
    H=H,
    Q=Q,
    R=R,
)

state = KalmanState(
    x=x[-1],
    P=P0,
)

In [ ]:
control_signal = z_estimated * 0
A_delay = np.linalg.matrix_power(A, delay)

# Commands sent so far, [u_{k-delay}, ..., u_{k-1}]. The DM has been holding the
# last integrator command since the telemetry loop stopped.
cmd_buffer = deque([np.copy(z_estimated[:nModes, 0])] * delay, maxlen=delay)
loop_times = []

for i in range(5000):

    z_output = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)[:nModes, 0]
    t0 = time.perf_counter()

    # Convert residual measurement into an atmospheric measurement,
    # x_k = z_k + u_{k-delay}, the same as estimate_atmospheric_state
    x_measurement = cmd_buffer[0] + z_output

    # Estimate atmosphere
    state, K = kalman_filter(
        measurement=x_measurement,
        state=state,
        matrices=matrices,
    )

    # The command computed now is first seen in z_{k+delay}: predict delay steps ahead
    z_estimated_kalman = A_delay @ state.x
    cmd_buffer.append(z_estimated_kalman)
    control_signal[:nModes, 0] = z_estimated_kalman

    if np.abs(z_estimated_kalman).max() > 10:
        CONTROL_SIGNAL_SHM.set_data(control_signal*0)
        print(i, "out")
        break
    
    CONTROL_SIGNAL_SHM.set_data(-control_signal)
    loop_times.append(time.perf_counter() - t0)
CONTROL_SIGNAL_SHM.set_data(control_signal*0)

loop_times = np.array(loop_times) * 1e3
print(f"compute time per frame: mean {loop_times.mean():.3f} ms, max {loop_times.max():.3f} ms")